In [54]:
base_scale = 1.00
step_scale = 0.30

# REAL TEST

In [55]:
import numpy as np
import cvxpy as cp
import torch
import joblib
from pathlib import Path
from models.models import MTLSharedHeads
import sys, os, andes

ROOT_DIR = Path.cwd()
if (ROOT_DIR / 'experiments').is_dir() and (ROOT_DIR / 'scheduling').is_dir():
    pass
elif ROOT_DIR.name == 'experiments' and (ROOT_DIR.parent / 'scheduling').is_dir():
    ROOT_DIR = ROOT_DIR.parent

if ROOT_DIR != Path.cwd():
    os.chdir(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

from experiments.run_sim_extract_ed import _load_yaml, _repeat_or_validate, add_measurement_devices
from scheduling.mtlsh_relu_convex import compute_feature_bounds_from_training_data
from data_generation.extract_metrics import build_feature_row

In [56]:
config_path = ROOT_DIR / 'experiments' / 'generation.yaml'
cost_config_path = ROOT_DIR / 'scheduling' / 'mtlsh_convex.yaml'

cfg = _load_yaml(Path(config_path))
cost_cfg = _load_yaml(Path(cost_config_path))
if 'ed_costs' not in cost_cfg:
    raise KeyError('Missing ed_costs in cost-config YAML.')

In [57]:
model_cfg = cost_cfg["model"]
state_path = Path(model_cfg["state_dict"])
if state_path.is_dir():
    state_path = state_path / "vis_mlp_state_dict.pt"

model = MTLSharedHeads(
    in_dim=int(model_cfg["in_dim"]),
    n_tasks=int(model_cfg["n_tasks"]),
    shared_sizes=model_cfg["shared_sizes"],
    head_sizes=model_cfg["head_sizes"],
    dropout=float(model_cfg.get("dropout", 0.0)),
)
state = torch.load(state_path, map_location="cpu")
model.load_state_dict(state)
model.eval()

def _linear_layers(module):
    return [m for m in module if isinstance(m, torch.nn.Linear)]

shared_layers = [
    (m.weight.detach().cpu().numpy(), m.bias.detach().cpu().numpy())
    for m in _linear_layers(model.shared)
]
head_layers = [
    [
        (m.weight.detach().cpu().numpy(), m.bias.detach().cpu().numpy())
        for m in _linear_layers(head)
    ]
    for head in model.heads
]

Add feature bounds based on training data for neuron activation (big-M limits)

In [58]:
# x_min/x_max must be in the SAME scale used by the NN
x_min, x_max, _ = compute_feature_bounds_from_training_data(cost_cfg)

x_min = np.asarray(x_min, dtype=float)
x_max = np.asarray(x_max, dtype=float)

assert x_min.shape[0] == model_cfg["in_dim"]
assert x_max.shape[0] == model_cfg["in_dim"]

Add bounds on predicted labels (i.e. prod. of IBRs, freq. labels...)

In [59]:
def relu_big_m(z, y, a, z_min, z_max):
    # Exact ReLU with binary a
    return [
        y >= 0,
        y >= z,
        y <= z - z_min * (1 - a),
        y <= z_max * a,
    ]

x = cp.Variable(model_cfg["in_dim"], name="features")
constraints = []

# interval bounds for inputs
h_min = x_min.copy()
h_max = x_max.copy()
h = x

# shared trunk
for li, (W, b) in enumerate(shared_layers):
    z = W @ h + b
    z_min = np.maximum(W, 0) @ h_min + np.minimum(W, 0) @ h_max + b
    z_max = np.maximum(W, 0) @ h_max + np.minimum(W, 0) @ h_min + b

    y = cp.Variable(b.shape[0], name=f"shared_{li}")
    a = cp.Variable(b.shape[0], boolean=True, name=f"shared_bin_{li}")
    constraints += relu_big_m(z, y, a, z_min, z_max)

    h = y
    h_min = np.maximum(0, z_min)
    h_max = np.maximum(0, z_max)

# heads
outputs = []
for head_idx, layers in enumerate(head_layers):
    h_head = h
    hmin_head = h_min
    hmax_head = h_max

    for li, (W, b) in enumerate(layers):
        z = W @ h_head + b
        if li < len(layers) - 1:
            z_min = np.maximum(W, 0) @ hmin_head + np.minimum(W, 0) @ hmax_head + b
            z_max = np.maximum(W, 0) @ hmax_head + np.minimum(W, 0) @ hmin_head + b

            y = cp.Variable(b.shape[0], name=f"head{head_idx}_{li}")
            a = cp.Variable(b.shape[0], boolean=True, name=f"head{head_idx}_bin_{li}")
            constraints += relu_big_m(z, y, a, z_min, z_max)

            h_head = y
            hmin_head = np.maximum(0, z_min)
            hmax_head = np.maximum(0, z_max)
        else:
            y_out = cp.Variable(1, name=f"out{head_idx}")
            constraints.append(y_out == z)
            outputs.append(y_out)

y = cp.hstack(outputs)

# output bounds (scaled space)
y_min = np.asarray(cost_cfg["bounds"]["y_min"], dtype=float)
y_max = np.asarray(cost_cfg["bounds"]["y_max"], dtype=float)
constraints += [y >= y_min, y <= y_max]

/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/cvxpy/expressions/expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 145 times so far.

  warnings.warn(msg, UserWarning)
/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/cvxpy/expressions/expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 146 times so far.

  warn

Give the features values as constraints

In [60]:
case_path = cfg['case']
ss = andes.load(case_path, setup=False)
ss.config.freq = float(50)
add_measurement_devices(ss)

rng = np.random.default_rng(int(cfg.get('seed', 42)))
regcv1_ids = ss.REGCV1.name.v
M_vec = rng.uniform(cfg['ibr']['M_range'][0], cfg['ibr']['M_range'][1], size=len(regcv1_ids))
D_vec = rng.uniform(cfg['ibr']['D_range'][0], cfg['ibr']['D_range'][1], size=len(regcv1_ids))

for uid in range(ss.PQ.n):
    ss.PQ.p0.v[uid] = ss.PQ.p0.v[uid] * base_scale
    ss.PQ.q0.v[uid] = ss.PQ.q0.v[uid] * base_scale
for uid in range(ss.PV.n):
    ss.PV.p0.v[uid] = ss.PV.p0.v[uid] * base_scale
    ss.PV.q0.v[uid] = ss.PV.q0.v[uid] * base_scale

ss.REGCV1.M.v, ss.REGCV1.D.v = M_vec, D_vec

ss.PQ.config.p2p = 1
ss.PQ.config.q2q = 1
ss.PQ.config.p2z = 0
ss.PQ.config.q2z = 0
ss.PQ.config.p2i = 0
ss.PQ.config.q2i = 0
ss.PQ.config.pq2z = 0

pq_p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy()
pq_q_before = np.asarray(ss.PQ.q0.v, dtype=float).copy()
pq_names = list(ss.PQ.name.v) if ss.PQ.n else []

for uid in range(ss.PQ.n):
    ss.PQ.p0.v[uid] = ss.PQ.p0.v[uid] * step_scale
    ss.PQ.q0.v[uid] = ss.PQ.q0.v[uid] * step_scale
pq_p_after = np.asarray(ss.PQ.p0.v, dtype=float).copy()
pq_q_after = np.asarray(ss.PQ.q0.v, dtype=float).copy()

M_agg = np.mean(np.concatenate([ss.GENROU.M.v, ss.REGCV1.M.v])).sum()
D_agg = np.mean(np.concatenate([ss.GENROU.D.v, ss.REGCV1.D.v])).sum()

features = build_feature_row(
    base_load_scale=base_scale,
    load_step_scale=step_scale,
    load_step_time=float(cfg['tds']['load_step_time']),
    pq_names=pq_names,
    pq_p_before=pq_p_before,
    pq_q_before=pq_q_before,
    pq_p_after=pq_p_after,
    pq_q_after=pq_q_after,
    M_vec=M_vec,
    D_vec=D_vec,
    M_agg=M_agg,
    D_agg=D_agg,
)

x_features = cost_cfg.get('features', {}).get('x_features') or list(features.keys())

In [61]:
ss.setup()

ng = ss.PV.n + ss.Slack.n
Pg = cp.Variable(ng)
Pd = float(np.sum(ss.PQ.p0.v))

Pg_min = ss.PV.pmin.v.tolist() + ss.Slack.pmin.v.tolist()
Pg_max = ss.PV.pmax.v.tolist() + ss.Slack.pmax.v.tolist()
Pg_now = ss.PV.p0.v.tolist() + ss.Slack.p0.v.tolist()

constraints += [
    cp.sum(Pg) == Pd,
    Pg >= Pg_min,
    Pg <= Pg_max,
]

In [62]:
print("We will produce: ", Pd)
print("We are producing: ", np.sum(Pg_now))
print("Which means a difference of: ", Pd-np.sum(Pg_now), "or of ", (Pd/np.sum(Pg_now)-1)*100,"%")

We will produce:  17.569199999999995
We are producing:  58.94561092000001
Which means a difference of:  -41.37641092000001 or of  -70.1942184909329 %


In [63]:
# If you want to fix x to current scaled features:
feat_vec = np.array([features[name] for name in x_features], dtype=float).reshape(1, -1)
x_scaler_path = cost_cfg.get("scalers", {}).get("x_scaler_path")
if x_scaler_path:
    x_scaler = joblib.load(x_scaler_path)
    feat_scaled = x_scaler.transform(feat_vec)
else:
    feat_scaled = feat_vec

constraints += [x == feat_scaled.reshape(-1)]

/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


link outputs of IBRs with real Pg val.

In [64]:
# example: y[4:8] are extra power for 4 IBRs, mapped to Pg indices
ibr_idx = [0, 5, 7, 8]   # <- indices of IBRs in Pg
P_extra = y[4:8]         # NN outputs

# extra power cannot exceed headroom
Pg_min = np.asarray(Pg_min, dtype=float)
Pg_max = np.asarray(Pg_max, dtype=float)
Pg_now = np.asarray(Pg_now, dtype=float)
constraints += [P_extra == Pg_now[ibr_idx] - Pg[ibr_idx]]


# Baseline ED (no NN constraints)
Run plain economic dispatch to compare against NN-constrained solve.


In [65]:
ng = ss.PV.n + ss.Slack.n
Pg = cp.Variable(ng)
Pd = float(np.sum(ss.PQ.p0.v)) * 0.9

Pg_min = np.asarray(ss.PV.pmin.v.tolist() + ss.Slack.pmin.v.tolist(), dtype=float)
Pg_max = np.asarray(ss.PV.pmax.v.tolist() + ss.Slack.pmax.v.tolist(), dtype=float)

a = _repeat_or_validate(cost_cfg['ed_costs']['a'], ng, 'ed_costs.a')
b = _repeat_or_validate(cost_cfg['ed_costs']['b'], ng, 'ed_costs.b')
c = _repeat_or_validate(cost_cfg['ed_costs']['c'], ng, 'ed_costs.c')

cost_expr = a + cp.multiply(b, Pg) + cp.multiply(c, cp.square(Pg))
objective = cp.Minimize(cp.sum(cost_expr))

constraints_ed = [
    cp.sum(Pg) == Pd,
    Pg >= Pg_min,
    Pg <= Pg_max,
]

prob_ed = cp.Problem(objective, constraints_ed)
prob_ed.solve(solver='GUROBI')
print('ED status:', prob_ed.status)
print('ED cost:', prob_ed.value)
print('Pg_opt:', Pg.value)
Pg_baseline = Pg.value.copy()


ED status: optimal
ED cost: 169.5321473284295
Pg_opt: [5.08718925 0.05871893 0.05871893 0.05871893 0.05871893 5.08718925
 0.05871893 5.08718925 0.19839866 0.05871896]


# ED + convex NN relaxation (coupled)
Couple NN outputs to Pg so constraints can actually change the ED result.
Here we tie y[4:8] (IBR extra power) to deviations from baseline Pg.


In [66]:
from scheduling.mtlsh_relu_convex import build_mtlsh_convex_constraints

# Build convex NN constraints (ReLU epigraph)
x_nn, y_nn, nn_constraints = build_mtlsh_convex_constraints(cost_cfg)

# Fix NN input to current scaled features (can be replaced by a Pg->x mapping)
feat_vec = np.array([features[name] for name in x_features], dtype=float).reshape(1, -1)
x_scaler_path = cost_cfg.get('scalers', {}).get('x_scaler_path')
if x_scaler_path:
    x_scaler = joblib.load(x_scaler_path)
    feat_scaled = x_scaler.transform(feat_vec)
else:
    feat_scaled = feat_vec

nn_constraints = list(nn_constraints)
nn_constraints.append(x_nn == feat_scaled.reshape(-1))

# New ED variable for NN-constrained solve
Pg_nn = cp.Variable(ng)
cost_expr_nn = a + cp.multiply(b, Pg_nn) + cp.multiply(c, cp.square(Pg_nn))
objective_nn = cp.Minimize(cp.sum(cost_expr_nn))

constraints_ed_nn = [
    cp.sum(Pg_nn) == Pd,
    Pg_nn >= Pg_min,
    Pg_nn <= Pg_max,
]

# Example coupling: y[4:8] = extra power from IBRs
ibr_idx = [0, 5, 7, 8]  # update to your actual IBR indices
P_extra = y_nn[4:8]

Pg_now = Pg_baseline
constraints_ed_nn += [P_extra == Pg_now[ibr_idx] - Pg_nn[ibr_idx]]

constraints_nn = nn_constraints + constraints_ed_nn

prob_nn = cp.Problem(objective_nn, constraints_nn)
prob_nn.solve(solver='GUROBI')
print('ED+NN status:', prob_nn.status)
print('ED+NN cost:', prob_nn.value)
print('Pg_opt (NN):', Pg_nn.value)
print('y_nn:', y_nn.value)


ED+NN status: optimal
ED+NN cost: 169.5321473284295
Pg_opt (NN): [5.08718926 0.05871893 0.05871893 0.05871893 0.05871893 5.08718926
 0.05871893 5.08718926 0.19839866 0.05871893]
y_nn: [-5.00000000e-02 -5.00000000e-01 -2.00000000e-01 -2.00000000e-02
 -1.09164304e-08 -1.09284075e-08 -1.18260308e-08 -1.36050729e-09]


/Applications/anaconda3/envs/GESTURE/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
